# DeepTuneIO — Module 05: Darshan Parser Processor

Uses the common `DATASET_FILTER` model shared with the DXT processor: `exact`, `prefix`, `extension`, `regex`, with one or more accepted values.


## Nuevo producto: Execution Summary

Module 05 conserva todas las salidas existentes y añade:

`4_Execution_Summary_*.csv`

Este CSV contiene exactamente una fila por `Jobid`. Para campañas `file_per_process` (NPZ y TFRecord), agrega los contadores POSIX entre todos los ficheros del mismo experimento. Las configuraciones deben ser consistentes; los timestamps usan el mínimo de inicio y máximo de fin; y ACCESS/STRIDE se reconstruyen globalmente por frecuencia. Esta será la entrada Parser del Module 09.


In [3]:
from pathlib import Path
import sys
import pandas as pd
from importlib.util import spec_from_file_location, module_from_spec

MODULE_DIR = Path("./Module").resolve()
MODULE_FILE = MODULE_DIR / "05_process_parser_reports.py"

spec = spec_from_file_location("deeptuneio_module05", MODULE_FILE)
module05 = module_from_spec(spec)
sys.modules[spec.name] = module05
spec.loader.exec_module(module05)

print("Module:", MODULE_FILE)
print("Version:", module05.VERSION)


Module: F:\NoteBooks\CORA_DR_UAB\Tesis\Scripts\DeepTuneIO\05_Aggregation\Module\05_process_parser_reports.py
Version: 4.2.0


## 1. Selección de campaña y filtro


In [6]:
CAMPAIGNS = {
    "DeepGalaxy": {
        "DG_bw512_f3c5x64": {
            "scenarios": [
                "lustre_1ost",
                "lustre_2ost",
                "lustre_4ost",
                "nfs_SnGPU_e1",
            ],
            "file_format": ".hdf5",
            "dataset_filter": {
                "mode": "exact",
                "values": [
                    "DeepGalaxy_bw512_f3c5x64_o.hdf5",
                    "DG_bw512_original_f3c5x64.hdf5",
                    "output_bw_512.hdf5",
                ],
            },
        },
    },

    "DLIOv1": {
        "HDF5_64bs": {
            "scenarios": ["lustre_ost"],
            "file_format": ".h5",
            "dataset_filter": {
                "mode": "prefix",
                "values": ["img_"],
            },
        },
        "NPZ_64bs": {
            "scenarios": ["lustre_ost"],
            "file_format": ".npz",
            "dataset_filter": {
                "mode": "prefix",
                "values": ["img_"],
            },
        },
        "TFRecord_256kts_64bs": {
            "scenarios": ["lustre_ost"],
            "file_format": ".tfrecords",
            "dataset_filter": {
                "mode": "prefix",
                "values": ["img_"],
            },
        },
        "TFRecord_1mts_64bs": {
            "scenarios": ["lustre_ost"],
            "file_format": ".tfrecords",
            "dataset_filter": {
                "mode": "prefix",
                "values": ["img_"],
            },
        },
    },
}


APPLICATION_POS = 1
DATASET_POS = 1
SCENARIO_POS = 4

APPLICATION = list(CAMPAIGNS)[APPLICATION_POS - 1]
DATASET_GROUP = list(CAMPAIGNS[APPLICATION])[DATASET_POS - 1]
SCENARIO = CAMPAIGNS[APPLICATION][DATASET_GROUP]["scenarios"][SCENARIO_POS - 1]
FILE_FORMAT = CAMPAIGNS[APPLICATION][DATASET_GROUP]["file_format"]
DATASET_FILTER = CAMPAIGNS[APPLICATION][DATASET_GROUP]["dataset_filter"]

print("Application   :", APPLICATION)
print("Dataset group :", DATASET_GROUP)
print("Scenario      :", SCENARIO)
print("File format   :", FILE_FORMAT)
print("Filter mode   :", DATASET_FILTER["mode"])
print("Filter values :", DATASET_FILTER["values"])


Application   : DeepGalaxy
Dataset group : DG_bw512_f3c5x64
Scenario      : nfs_SnGPU_e1
File format   : .hdf5
Filter mode   : exact
Filter values : ['DeepGalaxy_bw512_f3c5x64_o.hdf5', 'DG_bw512_original_f3c5x64.hdf5', 'output_bw_512.hdf5']


In [7]:
PROJECT_ROOT = Path(r"D:\DL_Experiments\CORA_DR_UAB").resolve()
ARTICLE_ID = "Tesis"
CAMPAIGN_ROOT = PROJECT_ROOT / ARTICLE_ID / "Data" / APPLICATION / DATASET_GROUP / SCENARIO

print("Campaign root:", CAMPAIGN_ROOT)
print("Exists       :", CAMPAIGN_ROOT.exists())


Campaign root: D:\DL_Experiments\CORA_DR_UAB\Tesis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1
Exists       : True


## 2. Ejecutar Module 05


In [8]:
return_code = module05.main([
    "--campaign-root", str(CAMPAIGN_ROOT),
    "--application", APPLICATION,
    "--dataset-group", DATASET_GROUP,
    "--scenario", SCENARIO,
    "--file-format", FILE_FORMAT,
    "--filter-mode", DATASET_FILTER["mode"],
    "--filter-values",
    *DATASET_FILTER["values"],
    "--event", "Tesis",
    "--module-root", str(MODULE_DIR),
])

print("Return code:", return_code)


DeepTuneIO — Module 05: Darshan Parser Processor
Campaign root : D:\DL_Experiments\CORA_DR_UAB\Tesis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1
Application   : DeepGalaxy
Dataset group : DG_bw512_f3c5x64
Scenario      : nfs_SnGPU_e1
File format   : .hdf5
Filter mode   : exact
Filter values : ['DeepGalaxy_bw512_f3c5x64_o.hdf5', 'DG_bw512_original_f3c5x64.hdf5', 'output_bw_512.hdf5']
Parser input  : D:\DL_Experiments\CORA_DR_UAB\Tesis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1\processed\Parser
Parser output : D:\DL_Experiments\CORA_DR_UAB\Tesis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1\results\Parser


Procesando DeepGalaxy:   0%|          | 0/25 [00:00<?, ?it/s]

[Pivot] Transformacion a matriz completada.
[4b] Archivo depurado y clasificado: D:\DL_Experiments\CORA_DR_UAB\Tesis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1\results\Parser\4b_Analisis_IO_DeepGalaxy_Parser_nfs_SnGPU_e1_DG_bw512_f3c5x64_Tesis_dep.csv
Execution summary: D:\DL_Experiments\CORA_DR_UAB\Tesis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1\results\Parser\4_Execution_Summary_DeepGalaxy_Parser_nfs_SnGPU_e1_DG_bw512_f3c5x64_Tesis.csv (25 rows)
Parser processing completed.
Return code: 0


In [9]:
PARSER_RESULTS_DIR = CAMPAIGN_ROOT / "results" / "Parser"
execution_files = sorted(PARSER_RESULTS_DIR.glob("4_Execution_Summary_*.csv"))
if execution_files:
    execution_df = pd.read_csv(execution_files[-1])
    print("Execution Summary:", execution_files[-1].name)
    print("Rows         :", len(execution_df))
    print("Unique JobID :", execution_df["Jobid"].nunique())
    print("Coverage min :", execution_df["Files_Coverage"].min())
    display(execution_df.head())


Execution Summary: 4_Execution_Summary_DeepGalaxy_Parser_nfs_SnGPU_e1_DG_bw512_f3c5x64_Tesis.csv
Rows         : 25
Unique JobID : 25
Coverage min : 1.0


,Jobid,Nprocs,Access_Pattern_Detected,Access_Pattern_flag,Exe_Mode_Flag,Access_Mode,Dataset_Selection,Num_Camera,Dataset_Path,Logical_Samples,...,POSIX_ACCESS4_ACCESS,POSIX_ACCESS4_COUNT,POSIX_STRIDE1_STRIDE,POSIX_STRIDE1_COUNT,POSIX_STRIDE2_STRIDE,POSIX_STRIDE2_COUNT,POSIX_STRIDE3_STRIDE,POSIX_STRIDE3_COUNT,POSIX_STRIDE4_STRIDE,POSIX_STRIDE4_COUNT
0,4942237,4,Shared,Shared,-m 0,0,s_*,14,output_bw_512.hdf5,35784,...,142,2016,1536,576,272,352,88,152,7887876,144
1,4948089,4,Shared+Reload+Shuffle,Shared+Reload+Shuffle,-m 1,1,s_*,14,output_bw_512.hdf5,35784,...,328,2515,272,1280,1536,576,88,152,7887876,144
2,4998168,8,Shared,Shared,-m 0,0,s_*,14,./dataset/output_bw_512.hdf5,35784,...,142,4032,1536,1152,272,574,88,304,7887876,288
3,5006144,48,Shared,Shared,-m 0,0,s_*,14,./dataset/output_bw_512.hdf5,35784,...,3656,9918,1536,6912,272,3024,88,1824,7887876,1728
4,5124492,8,Shared+Reload+Shuffle,Shared+Reload+Shuffle,-m 1,1,s_*,14,./dataset/output_bw_512.hdf5,35784,...,328,5008,272,2543,1536,1152,88,304,7887876,288
